# Quadratic elements and native Gmsh formats

`mesh_order=2` produces 6-node triangles (Gmsh type 9). Connectivity is stored
with 6 columns; `plot_by_grain` fills the first 3 corners.

**Abaqus INP** (`export_confmesh2d_inp`) currently writes linear CPS3/CPS4 from
those corners — use Gmsh `formats=['msh', 'vtk']` when you need the mid-side
nodes. Native write happens inside `femesh_gmsh` while the Gmsh session is live
(`mesh_gs(..., out_dir=..., formats=...)`).

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
}

In [ ]:
out = Path.cwd() / 'confMesh2d_quadratic_out'
lin = mesh_gs(cells, mesh_size_gb=0.4, mesh_size_bulk=0.8,
              mesh_order=1, mesh_algo=6, recombine_to_quads=False)
q2 = mesh_gs(cells, mesh_size_gb=0.4, mesh_size_bulk=0.8,
             mesh_order=2, mesh_algo=6, recombine_to_quads=False,
             out_dir=str(out), formats=['msh', 'vtk'], basename='rve_q2')
mq = q2['mesher']
mq.form_elsets_gmsh(); mq.build_boundary_nsets(); mq.build_gb_nset()
print('linear nodes', lin['n_nodes'], 'quadratic nodes', q2['n_nodes'])
print('quadratic conn', mq.elConn['triangle'].shape, 'order', mq.elementOrder)
print('native files', q2['exported'])
print(mq.validation_report)

In [ ]:
fig, ax = mq.plot_by_grain(figsize=(6, 6), show_gb=True, show_nsets=True,
                           title='quadratic tris (corners plotted)')
fig

In [ ]:
inp = mq.export_abaqus_inp(out / 'rve_cps3_from_corners.inp', plane='stress')
inp, summarize_inp(inp), q2['exported']